In [2]:
# Required imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

import seaborn as sns
from matplotlib import pyplot as plt

%matplotlib inline

In [3]:
# load the data
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

df.shape

(5000, 9)

In [4]:
# first look
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
# first look
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
# Data preparation
# The task asks to check for missing values and, if there are any, fill categorical features with 'NA' and numerical features with 0.0.

# missing values
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [7]:
# split the columns by type

categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical = [c for c in df.select_dtypes(include='number').columns if c != 'converted']

print("categorical:", categorical)
print("numerical:  ", numerical)

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical:   ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [8]:
# fill missing values

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()

np.int64(0)

In [9]:
'''
Question 1
What is the most frequent observation (mode) for the column industry?

- NA
- technology <-- answer
- healthcare
- retail
'''

# Step 1: count every industry
print(f"df.industry.value_counts() :\n{df.industry.value_counts()} \n") 

# Step 2: take the mode directly
print(f"df.industry.mode() :\n{df.industry.mode()} \n")

# Step 3: see how close the race is (optional but useful)
print(f"df.industry.value_counts(normalize=True).round(3) :\n{df.industry.value_counts(normalize=True).round(3)}") # normalize=True shows shares instead of counts.

df.industry.value_counts() :
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64 

df.industry.mode() :
0    technology
Name: industry, dtype: str 

df.industry.value_counts(normalize=True).round(3) :
industry
technology       0.235
retail           0.185
healthcare       0.168
finance          0.144
education        0.128
manufacturing    0.093
NA               0.048
Name: proportion, dtype: float64


In [14]:
'''
# Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score <-- answer
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

Split the data:

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

Make sure that the target value converted is not in your dataframe.
'''

# Step 1: confirm the inputs
print(f"numerical:\n{numerical}")
df[numerical].isnull().sum()

# Step 2: the correlation matrix
corr = df[numerical].corr()
corr.round(3)

# Step 3: only the four pairs from the question
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

results = {(a, b): corr.loc[a, b] for a, b in pairs}

print("\n")

for (a, b), value in results.items():
    print(a, '|', b, '|', round(value, 4))

print("\n")

# Step 4: pick the largest
best_pair = max(results, key=results.get)
print(f"best_pair:\n{best_pair}, {round(results[best_pair], 4)}")

numerical:
['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


interaction_count | lead_score | 0.9157
number_of_courses_viewed | lead_score | 0.7572
number_of_courses_viewed | interaction_count | 0.7216
annual_income | interaction_count | 0.1228


best_pair:
('interaction_count', 'lead_score'), 0.9157


In [19]:
'''
# Question 3
Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source <- answer
- employment_status
'''

# Split the data
# The homework gives the exact calls, so use them as written. The split happens after the fill, so every part is clean.
# Step 1: split - This is the 60/20/20 split, because 25% of the remaining 80% is 20% of the total.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

len(df_train), len(df_val), len(df_test)

# Step 2: clean the indexes and pull out the target
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

# The homework says the target must not stay in the DataFrame. That’s why we keep it as the separate arrays y_train, y_val and y_test.
del df_train['converted']
del df_val['converted']
del df_test['converted']

# Question 3: mutual information with converted

# Checks that belong to this question
'''
Check	                                 Why it matters
Use only the training set	             The task says so, and it avoids leaking validation information
Use only the categorical columns	     MI as we use it works on categories
Use y_train as the target	             converted was removed from df_train
Round with round(score, 2)	             The answer options are based on 2 decimals
'''

print(f"converted in df_train.columns:{'converted' in df_train.columns}\n")   # expect False
print(f"len(df_train):{len(df_train)}, len(y_train): {len(y_train)}\n")        # expect 3000 3000
print(f"categorical:\n{categorical}\n")

# Step 1: the scores - 
# Print the rounded score, which is the one the question uses, next to a more precise one. The extra digits only help you see how close the values are.
mi = {}

for c in categorical:
    mi[c] = mutual_info_score(df_train[c], y_train)

for c, score in sorted(mi.items(), key=lambda x: x[1], reverse=True):
    print(c, '|', round(score, 2), '|', round(score, 6))

print("\n")
'''
What to watch for:

- If two variables tie after rounding to 2 decimals, the longer numbers show which is really larger, and I’d like to see both before we decide.
- MI is symmetric, so the order of the two arguments doesn’t change the result.
'''

# Step 2: pick the winner
best = max(mi, key=mi.get)
print(best, round(mi[best], 2))


converted in df_train.columns:False

len(df_train):3000, len(y_train): 3000

categorical:
['lead_source', 'industry', 'employment_status', 'location']

lead_source | 0.03 | 0.034346
employment_status | 0.02 | 0.018966
industry | 0.0 | 0.001995
location | 0.0 | 0.001043


lead_source 0.03


In [22]:
'''
# Question 4: train a logistic regression and get the validation accuracy
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

0.55
0.65 <-- answer
0.75
0.85
'''

'''
Checks that belong to this question
Check	                                                            Why it matters
Features are all four categorical and all four numerical columns	The task says to use the categorical ones with one-hot encoding, and the numeric ones go in as they are
fit_transform on train, only transform on validation	            Validation must have exactly the training columns (lesson 3.8)
The exact model parameters from the homework	                    Results stay reproducible across Scikit-Learn versions
Accuracy on validation, not train	                                Validation is the honest estimate
Keep the unrounded accuracy	                                        Q5 needs it without rounding
'''

# Step 1: one-hot encoding
features = categorical + numerical

train_dict = df_train[features].to_dict(orient='records')
val_dict = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

'''
Check the output: 
- Both matrices should have the same number of columns, with 3,000 and 1,000 rows. 
- The names should look like lead_source=referral for categories and plain names like annual_income for numbers.
'''

print(f"X_train.shape:{X_train.shape}, X_val.shape:{X_val.shape}\n")
print(f"list(dv.get_feature_names_out()):{list(dv.get_feature_names_out())}\n")

# Step 2: train with the exact parameters
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Step 3: predict on validation and compute accuracy
y_pred = model.predict_proba(X_val)[:, 1] # predict_proba(...)[:, 1] is the probability of converted = 1, and >= 0.5 turns it into a yes/no decision.
decision = (y_pred >= 0.5) # The mean of True/False values is the share of correct predictions.

accuracy = (decision == y_val).mean()

print(f"unrounded accuracy: {accuracy}\n")
print(f"rounded accuracy: {round(accuracy, 2)}\n")

# Step 4: a sanity check (not part of the answer)
# The second number is the accuracy of a model that ignores the features and always predicts the more common class. If your model barely beats it, accuracy alone says little, which is why week 4 introduces other metrics.
print(f'share converted in y_val: {y_val.mean()}\n')
print(f'always-guess-majority accuracy: {max(y_val.mean(), 1 - y_val.mean())}\n') # The model is only modestly useful. Its 0.645 compares with 0.597 for always guessing the majority class. That’s a small gain, which is why week 4 introduces better metrics than accuracy.

X_train.shape:(3000, 28), X_val.shape:(1000, 28)

list(dv.get_feature_names_out()):['annual_income', 'employment_status=NA', 'employment_status=employed', 'employment_status=self_employed', 'employment_status=student', 'employment_status=unemployed', 'industry=NA', 'industry=education', 'industry=finance', 'industry=healthcare', 'industry=manufacturing', 'industry=retail', 'industry=technology', 'interaction_count', 'lead_score', 'lead_source=NA', 'lead_source=events', 'lead_source=organic_search', 'lead_source=paid_ads', 'lead_source=referral', 'lead_source=social_media', 'location=NA', 'location=africa', 'location=asia', 'location=europe', 'location=north_america', 'location=south_america', 'number_of_courses_viewed']

unrounded accuracy: 0.645

rounded accuracy: 0.64

share converted in y_val: 0.597

always-guess-majority accuracy: 0.597



In [24]:
'''
# Question 5
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

'lead_source'
'number_of_courses_viewed' <-- answer
'interaction_count'
Note: The difference doesn't have to be positive.
'''

'''
Checks that belong to this question

Check	                                                            Why it matters
Same features and parameters as Q4	                                The baseline must reproduce 0.645, or the comparison is unfair
Remove one original column at a time (not one one-hot column)	    A “feature” here is lead_source, not lead_source=events
Refit DictVectorizer for each subset	                            The columns change when a feature is removed
Use the unrounded accuracy	                                        The task says without rounding. Differences between close models are tiny
Difference = original minus without	                                A positive value means removing the feature hurt the model
'''

# Step 1: one function that trains and scores any list of features
def train_and_score(feature_list):
    train_dict = df_train[feature_list].to_dict(orient='records')
    val_dict = df_val[feature_list].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    return float(((y_pred >= 0.5) == y_val).mean())

# Step 2: reproduce the Q4 baseline
features = categorical + numerical

original = train_and_score(features)
print(f"original accuracy: {original}\n") # This must print 0.645. same as q4

# Step 3: remove each feature, one at a time
results = []

for f in features:
    without = [x for x in features if x != f]
    score = train_and_score(without)
    results.append((f, score, original - score))

for f, score, diff in results:
    print(f'{f:28s} accuracy={score:.4f}  diff={diff:+.4f}') # diff is original - score. The + in the format shows the sign explicitly.

print("\n")

# Step 4: the three features from the question
'''
The question picks one of these three, even if another feature in the full list has a smaller difference. 
The note says the difference doesn’t have to be positive, which means a feature can even make the model slightly worse.
'''
asked = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for f, score, diff in results:
    if f in asked:
        print(f"Feature Excluded: {f}, accuracy difference: {round(diff, 4)}")

original accuracy: 0.645

lead_source                  accuracy=0.6420  diff=+0.0030
industry                     accuracy=0.6450  diff=+0.0000
employment_status            accuracy=0.6450  diff=+0.0000
location                     accuracy=0.6450  diff=+0.0000
annual_income                accuracy=0.7240  diff=-0.0790
number_of_courses_viewed     accuracy=0.6430  diff=+0.0020
interaction_count            accuracy=0.6010  diff=+0.0440
lead_score                   accuracy=0.6440  diff=+0.0010


Feature Excluded: lead_source, accuracy difference: 0.003
Feature Excluded: number_of_courses_viewed, accuracy difference: 0.002
Feature Excluded: interaction_count, accuracy difference: 0.044


In [26]:
'''
Question 6
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001 <-- answer

Note: If there are multiple options, select the smallest C.
'''

'''
Checks that belong to this question

Check	                                                                      Why it matters
All features from Q4 (4 categorical + 4 numerical)	                          The task says “all the features as in Q4”
Same other parameters (solver='liblinear', max_iter=1000, random_state=42)	  Only C should change
Round accuracy to 3 decimals	                                              Different from Q4, which used 2
Ties: pick the smallest C	                                                  Stated in the note
'''

'''
C is the inverse of the regularization strength, so a smaller C means stronger regularization. The values to try are all very small:

C in [0.000001, 0.00001, 0.0001, 0.001]
'''

# Step 1: confirm the matrices from Q4 are still the ones in memory
print(f"X_train.shape: {X_train.shape}, X_val.shape: {X_val.shape}\n")   # expect (3000, 28) (1000, 28)

# Step 2: train for each C
scores = {}

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    accuracy = float(((y_pred >= 0.5) == y_val).mean())

    scores[C] = round(accuracy, 3)
    print(f"C: {C}, accuracy: {accuracy}, scores[C]: {scores[C]}\n")

# Step 3: pick the best C, smallest on ties
best_score = max(scores.values())
best_C = min(C for C, s in scores.items() if s == best_score)

print(f"best_C: {best_C}, best_score: {best_score}")

X_train.shape: (3000, 28), X_val.shape: (1000, 28)

C: 1e-06, accuracy: 0.598, scores[C]: 0.598

C: 1e-05, accuracy: 0.598, scores[C]: 0.598

C: 0.0001, accuracy: 0.613, scores[C]: 0.613

C: 0.001, accuracy: 0.645, scores[C]: 0.645

best_C: 0.001, best_score: 0.645
